In [0]:
#Get paths and client key from the Databricks job's for-each task.
#Preferred: pass ONE task parameter  work_item = {{input}}  (the whole work item from Triage_File_Reader as JSON), so file types added
#later are picked up without changing the job. Individual parameters (wc_loss_run_path = {{input.wc_loss_run_path}}, etc.) still work.
import json
PATH_KEYS = ["wc_loss_run_path", "liab_loss_run_path", "ttd_path", "ptr_path", "wc_ptr_path", "liab_ptr_path"]
for k in ["work_item"] + PATH_KEYS + ["client_key"]:
   dbutils.widgets.text(k, "")

work_item_json = dbutils.widgets.get("work_item").strip()
if work_item_json:
   work_item = json.loads(work_item_json)
   paths      = {k: work_item.get(k, "") or "" for k in PATH_KEYS}
   client_key = work_item.get("client_key", "")
else:
   paths      = {k: dbutils.widgets.get(k) for k in PATH_KEYS}
   client_key = dbutils.widgets.get("client_key")

print("Client:", client_key)
for k in PATH_KEYS:
   print(f"  {k}: {paths[k] or '(none)'}")

In [0]:
#Define paths for various directories. Run mkdirs command to ensure directories exist
INPROGRESS_DIR = "/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/in_progress/"
PROCESSED_DIR  = "/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/processed"
FAILED_DIR     = "/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/failed"

for p in [INPROGRESS_DIR, PROCESSED_DIR, FAILED_DIR]:
   dbutils.fs.mkdirs(p)

In [0]:
#Define file-mover function
def move(src, dst_dir):
   dst = f"{dst_dir.rstrip('/')}/{src.split('/')[-1]}"
   dbutils.fs.mv(src, dst)
   return dst

In [0]:
#Moves files from input folder to in_progress folder as they are collected for each run. Prevents files from being processed multiple times
claimed = {}
for k, p in paths.items():
   if p:
       claimed[k] = move(p, INPROGRESS_DIR)

In [0]:
#Runs Transformation notebook using claimed files as inputs. If Transformation fails, files are moved to failed folder. If Transformation succeeds, files are moved to processed folder.
try:
   dbutils.notebook.run(
       "/Workspace/Users/jonah.crook@aon.com/Triage/Triage_Transformation_Output_Consultant_DQ_Loop",
       timeout_seconds=0,
       arguments={**{k: claimed.get(k, "") for k in PATH_KEYS}, "client_key": client_key}
   )
   for p in claimed.values():
       move(p, PROCESSED_DIR)
except Exception:
   for p in claimed.values():
       move(p, FAILED_DIR)
   raise